# 3.6 Object-Oriented Programming

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 3: Python for Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-03-python-for-machine-learning/03.06_object-oriented-programming.ipynb)

Companion code for Section 3.6 of the book. Run the cells in order from top to bottom.

**Example 3.6.1**

In [ ]:
class LinearRegression:
    '''A from-scratch linear regression model.'''

    def __init__(self, learning_rate=0.01, n_epochs=1000):
        '''Constructor: initialise hyperparameters and placeholder weights.'''
        self.lr       = learning_rate
        self.n_epochs = n_epochs
        self.weights  = None   # Set during fit()
        self.bias     = None
        self.history  = []     # Loss per epoch

    def fit(self, X, y):
        '''Train model using gradient descent.'''
        import numpy as np
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias    = 0.0

        for _ in range(self.n_epochs):
            y_pred = self.predict(X)
            errors = y_pred - y

            # Gradient descent update
            dw = (2/n_samples) * X.T @ errors
            db = (2/n_samples) * errors.sum()
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db

            loss = np.mean(errors**2)
            self.history.append(loss)

        return self   # Enables method chaining: model.fit(X,y).predict(X_new)

    def predict(self, X):
        '''Generate predictions.'''
        return X @ self.weights + self.bias

    def score(self, X, y):
        '''R² score.'''
        import numpy as np
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - y.mean())**2)
        return 1 - ss_res / ss_tot

    def __repr__(self):
        '''String representation - shown in REPL and print().'''
        return f'LinearRegression(lr={self.lr}, epochs={self.n_epochs})'

    def __len__(self):
        '''len(model) returns number of parameters.'''
        if self.weights is None: return 0
        return len(self.weights) + 1  # weights + bias

# Using the class
import numpy as np
np.random.seed(42)
X = np.random.randn(200, 3)
y = 2*X[:,0] - 1.5*X[:,1] + 3*X[:,2] + 5 + 0.1*np.random.randn(200)

model = LinearRegression(learning_rate=0.05, n_epochs=500)
print(model)                        # __repr__: LinearRegression(lr=0.05, epochs=500)
model.fit(X, y)                     # Train
print(f'R² = {model.score(X, y):.6f}')  # 0.999...
print(f'Weights: {np.round(model.weights, 3)}')
print(f'Bias:    {model.bias:.3f}')
print(f'Params:  {len(model)}')
print(f'Initial loss: {model.history[0]:.4f}')
print(f'Final loss:   {model.history[-1]:.6f}')

**Example 3.6.2**

In [ ]:
class RidgeRegression(LinearRegression):
    '''Linear regression with L2 regularisation (ridge).'''

    def __init__(self, learning_rate=0.01, n_epochs=1000, alpha=1.0):
        super().__init__(learning_rate, n_epochs)  # Call parent __init__
        self.alpha = alpha   # Regularisation strength

    def fit(self, X, y):
        '''Override fit to add L2 penalty.'''
        import numpy as np
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias    = 0.0

        for _ in range(self.n_epochs):
            y_pred = self.predict(X)  # Inherited from parent
            errors = y_pred - y

            # L2 penalty added to weight gradient
            dw = (2/n_samples) * X.T @ errors + 2*self.alpha*self.weights
            db = (2/n_samples) * errors.sum()
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db
            self.history.append(np.mean(errors**2))

        return self

    def __repr__(self):
        return f'RidgeRegression(lr={self.lr}, epochs={self.n_epochs}, alpha={self.alpha})'

# Polymorphism: both classes have the same interface
models = [
    LinearRegression(learning_rate=0.05, n_epochs=500),
    RidgeRegression(learning_rate=0.05,  n_epochs=500, alpha=0.1),
    RidgeRegression(learning_rate=0.05,  n_epochs=500, alpha=10.0),
]

for m in models:
    m.fit(X, y)
    print(f'{repr(m)}')
    print(f'  R²={m.score(X,y):.4f}  weights={np.round(m.weights,3)}')

# isinstance - check class membership
ridge = models[1]
print(f'\nisinstance(ridge, RidgeRegression): {isinstance(ridge, RidgeRegression)}')
print(f'isinstance(ridge, LinearRegression): {isinstance(ridge, LinearRegression)}')

**Example 3.6.3**

In [ ]:
class MLDataset:
    '''Custom dataset class - mirrors PyTorch Dataset pattern.'''

    def __init__(self, X, y, transform=None):
        self.X         = X
        self.y         = y
        self.transform = transform

    def __len__(self):
        return len(self.X)                    # len(dataset) = num samples

    def __getitem__(self, idx):
        x_sample = self.X[idx]
        if self.transform:
            x_sample = self.transform(x_sample)
        return x_sample, self.y[idx]          # Return (features, label) tuple

    def __repr__(self):
        return f'MLDataset(n_samples={len(self)}, n_features={self.X.shape[1]})'

    def __iter__(self):
        for i in range(len(self)):            # Make dataset iterable
            yield self[i]

# Usage
import numpy as np
X_data = np.random.randn(1000, 30)
y_data = np.random.randint(0, 2, 1000)

dataset = MLDataset(X_data, y_data)
print(dataset)                    # MLDataset(n_samples=1000, n_features=30)
print(f'Size: {len(dataset)}')    # 1000

# Indexing and slicing
x_0, y_0 = dataset[0]
print(f'First sample: features={x_0.shape}, label={y_0}')

# Mini-batch sampling
batch_idx = np.random.choice(len(dataset), size=32, replace=False)
batch_X   = np.array([dataset[i][0] for i in batch_idx])
batch_y   = np.array([dataset[i][1] for i in batch_idx])
print(f'Batch: X={batch_X.shape}, y={batch_y.shape}')